## Pandas Advanced Operations ? Interview Notes

These topics are essential for advanced Pandas, data analytics, and AI/ML interview preparation. The key is understanding what shape of data goes in, what operation happens, and what shape comes out.

### 1. Complex aggregation with `groupby()` and named aggregation

Aggregation means summarizing multiple rows into statistics such as sum, mean, count, minimum, and maximum.

### 2. Pivot table ? summarize data into a matrix

### 3. `pd.crosstab()` ? frequency tables and proportions


In [8]:
import pandas as pd

df = pd.DataFrame({
    "Department": ["IT", "IT", "HR", "HR", "IT"],
    "Employee": ["A", "B", "C", "D", "E"],
    "Sales": [100, 200, 150, 250, 300],
    "Hours": [5, 8, 6, 10, 7]
})

df


,Department,Employee,Sales,Hours
0,IT,A,100,5
1,IT,B,200,8
2,HR,C,150,6
3,HR,D,250,10
4,IT,E,300,7


In [9]:
# Basic aggregation
print(df.groupby("Department")["Sales"].sum())

# Multiple aggregations
print("\nMultiple aggregations:")
print(df.groupby("Department")["Sales"].agg(["sum", "mean", "min", "max", "count"]))

# Named aggregation
result = df.groupby("Department").agg(
    total_sales=("Sales", "sum"),
    average_sales=("Sales", "mean"),
    maximum_hours=("Hours", "max"),
    employee_count=("Employee", "count")
)

print("\nNamed aggregation:")
print(result)

# transform() example
print("\nTransform example:")
print(df.groupby("Department")["Sales"].transform("mean"))

# groupby multiple columns
print("\nGroup by multiple columns:")
print(df.groupby(["Department", "Employee"]).agg(
    total_sales=("Sales", "sum"),
    average_hours=("Hours", "mean")
))


Department
HR    400
IT    600
Name: Sales, dtype: int64

Multiple aggregations:
            sum   mean  min  max  count
Department                             
HR          400  200.0  150  250      2
IT          600  200.0  100  300      3

Named aggregation:
            total_sales  average_sales  maximum_hours  employee_count
Department                                                           
HR                  400          200.0             10               2
IT                  600          200.0              8               3

Transform example:
0    200.0
1    200.0
2    200.0
3    200.0
4    200.0
Name: Sales, dtype: float64

Group by multiple columns:
                     total_sales  average_hours
Department Employee                            
HR         C                 150            6.0
           D                 250           10.0
IT         A                 100            5.0
           B                 200            8.0
           E                 300        

In [10]:
# Pivot table
pd.pivot_table(
    df,
    index="Department",
    columns="Employee",
    values="Sales",
    aggfunc="sum",
    fill_value=0
)

# Pivot table with margins
pd.pivot_table(
    df,
    index="Department",
    values="Sales",
    aggfunc="sum",
    margins=True,
    margins_name="Grand Total"
)


,Sales
Department,
HR,400
IT,600
Grand Total,1000


In [11]:
# Crosstab
df2 = pd.DataFrame({
    "Department": ["IT", "IT", "HR", "HR", "IT"],
    "Gender": ["M", "F", "F", "M", "M"]
})

print(pd.crosstab(df2["Department"], df2["Gender"]))

print("\nWith margins:")
print(pd.crosstab(
    df2["Department"],
    df2["Gender"],
    margins=True,
    margins_name="Total"
))


Gender      F  M
Department      
HR          1  1
IT          1  2

With margins:
Gender      F  M  Total
Department             
HR          1  1      2
IT          1  2      3
Total       2  3      5


## Interview traps

- `agg()` summarizes groups; it does not normally preserve every original row.
- `transform()` returns results aligned with the original rows.
- `count` excludes missing values; `size` counts rows, including rows with missing values.
- Named aggregation creates explicit output names and avoids awkward multi-level column names.
- `fill_value=0` changes missing cells in the result, not in the original DataFrame.

Quick interview summary:
- Use `groupby()` when you want grouped summary statistics.
- Use `pivot_table()` when you want a matrix-like summary.
- Use `crosstab()` when you want category counts/frequencies.


In [12]:
wide = pd.DataFrame({
    "Student": ["A", "B"],
    "Math": [80, 90],
    "Science": [85, 95]
})
wide

,Student,Math,Science
0,A,80,85
1,B,90,95


In [13]:
long = wide.melt(
    id_vars="Student",
    var_name="Subject",
    value_name="Marks"
)

print(long)
s = wide.set_index("Student").stack()

print(s)
a = s.unstack()

print(a)

  Student  Subject  Marks
0       A     Math     80
1       B     Math     90
2       A  Science     85
3       B  Science     95
Student         
A        Math       80
         Science    85
B        Math       90
         Science    95
dtype: int64
         Math  Science
Student               
A          80       85
B          90       95


## 4. Parquet: a faster, compact way to store tables

Imagine a spreadsheet with many rows and columns. A CSV file is like writing every row out as plain text. Parquet stores the same table in a more organized, compressed format that computers can read efficiently.

### Simple analogy

- **CSV** is like a long shopping receipt: easy to read, but everything is written out line by line.
- **Parquet** is like neatly packed boxes, with similar kinds of values stored together. If you only need the `Sales` column, a tool can often read that column without loading every other column.

### Why use Parquet?

- **Columnar storage:** values from each column are stored together, useful when analyzing only a few columns.
- **Compression:** files are often smaller than equivalent CSV files (the actual size depends on the data).
- **Types are preserved:** numbers, dates, and other supported types are stored with their type information; CSV usually starts as plain text and needs type detection when loaded.
- **Useful for analytics:** common in data lakes, Spark, cloud analytics, and large data workflows.

### CSV and Parquet at a glance

| Feature | CSV | Parquet |
|---|---|---|
| Human-readable in a text editor | Yes | No, it is a binary format |
| Stores columns together | No | Yes |
| Often compact for analytics data | Not as much | Often more compact |
| Easy to share with basic tools | Very easy | Needs compatible software |
| Pandas support | Built in | Needs `pyarrow` or `fastparquet` |

### Important related terms

- **Schema:** the description of a table's columns and data types.
- **Columnar format:** stores data by columns instead of writing each full row together.
- **Partitioning:** splits data into folders by a column such as year or department. This can help when reading only one group from a large dataset. For a small table, partitioning is usually unnecessary.

### When should I choose each format?

Use **CSV** when you want a simple, human-readable file for sharing or opening in a spreadsheet. Use **Parquet** for larger datasets and repeated analysis, especially when you often read only some columns or need compact storage.

### Pandas requirement

Pandas needs an optional Parquet engine. If the example below reports that neither `pyarrow` nor `fastparquet` is installed, install PyArrow in the notebook environment by running `%pip install pyarrow`, then restart the kernel if prompted.

### Fix: install the Parquet engine

The `ImportError` means Pandas is installed, but it has no Parquet reader/writer engine yet. Run the next cell once in this notebook to install PyArrow into the environment used by this notebook. If VS Code asks to restart the kernel, restart it, then run the Parquet example again.

`%pip` is used instead of `!pip` so installation targets the active notebook environment.

In [14]:
%pip install pyarrow


   ---------------------------------------- 0.0/28.5 MB ? eta -:--:--
   ---------------------------------------- 0.3/28.5 MB ? eta -:--:--
   - -------------------------------------- 1.0/28.5 MB 4.0 MB/s eta 0:00:07
   -- ------------------------------------- 1.6/28.5 MB 2.9 MB/s eta 0:00:10
   -- ------------------------------------- 1.6/28.5 MB 2.9 MB/s eta 0:00:10
   -- ------------------------------------- 1.8/28.5 MB 1.8 MB/s eta 0:00:15
   -- ------------------------------------- 1.8/28.5 MB 1.8 MB/s eta 0:00:15
   -- ------------------------------------- 1.8/28.5 MB 1.8 MB/s eta 0:00:15
   -- ------------------------------------- 2.1/28.5 MB 1.4 MB/s eta 0:00:19
   --- ------------------------------------ 2.4/28.5 MB 1.3 MB/s eta 0:00:21
   --- ------------------------------------ 2.4/28.5 MB 1.3 MB/s eta 0:00:21
   --- ------------------------------------ 2.6/28.5 MB 1.2 MB/s eta 0:00:22
   ---- ----------------------------------- 2.9/28.5 MB 1.2 MB/s eta 0:00:22
   ---- -----

In [15]:
from pathlib import Path
import tempfile

sales = pd.DataFrame({
    "Department": ["Bakery", "Produce", "Bakery", "Produce"],
    "Item": ["Bread", "Apples", "Cake", "Bananas"],
    "Sales": [20, 35, 50, 25],
})

# Keep this demo self-contained: temporary files are removed when the cell finishes.
with tempfile.TemporaryDirectory() as temp_dir:
    temp_dir = Path(temp_dir)
    csv_path = temp_dir / "sales.csv"
    parquet_path = temp_dir / "sales.parquet"
    partitioned_path = temp_dir / "sales_by_department"

    # CSV: simple text format
    sales.to_csv(csv_path, index=False)
    csv_loaded = pd.read_csv(csv_path)

    # Parquet: compact columnar format
    sales.to_parquet(parquet_path, index=False)
    parquet_loaded = pd.read_parquet(parquet_path)

    print("CSV file size:", csv_path.stat().st_size, "bytes")
    print("Parquet file size:", parquet_path.stat().st_size, "bytes")
    print("Data read from Parquet:")
    print(parquet_loaded)

    # Read only the columns needed for a question: less data to load on large files.
    print("\nOnly Department and Sales:")
    print(pd.read_parquet(parquet_path, columns=["Department", "Sales"]))

    # Partitioning makes folders by Department. Useful for larger datasets,
    # but usually unnecessary for a tiny table like this one.
    sales.to_parquet(partitioned_path, index=False, partition_cols=["Department"])
    print("\nRead the partitioned Parquet dataset:")
    print(pd.read_parquet(partitioned_path))

ArrowKeyError: No type extension with name arrow.py_extension_type found